# WaveLab RL trên Colab
Nếu cell kiểm tra Python không in thông báo, hãy kiểm tra nút Connect/Kết nối của Colab và runtime GPU. Chạy từng cell theo thứ tự; không Run all vì cell STOP ở cuối sẽ yêu cầu dừng train.


## Đối chứng reward vật lý — 14/11/2026
Chọn EXPERIMENT ở cell Cấu hình: physics_blur (mặc định), physics_nearest, physics_le2024 hoặc parameter_progress.
Ba nhánh physics dùng r=-J-0.01 và param_weight=0; nhãn chỉ phục vụ báo cáo. Baseline dùng r=2(e_trước-e_sau)-0.01.
Giữ cùng 256 train / 32 validation riêng, seed2026, 4k bước, batch8, 4 workers, encoder epoch50 đóng băng.
Mỗi nhánh lưu thư mục physics_compare_v1 riêng và bắt đầu từ checkpoint0; không dùng checkpoint/replay của reward cũ.
Đây là pilot một seed. Chưa có kết quả để kết luận nhánh nào tốt hơn. Le2024 ở đây là bản chuyển sang ảnh raster, chưa phải tái lập độc lập bài báo.
Chạy từng cell; không Run all vì cell STOP cuối notebook sẽ dừng train.


In [ ]:
print('Python đang hoạt động. Tiếp theo sẽ yêu cầu kết nối Google Drive.', flush=True)
from google.colab import drive
drive.mount('/content/drive')
print('Đã kết nối Google Drive.', flush=True)

In [ ]:
import subprocess,sys,os,json,hashlib,shutil,time
from pathlib import Path
REPO=Path('/content/wavelab-rl-scm')
def run_visible(command):
    print('Đang chạy:', ' '.join(map(str,command)), flush=True)
    child=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in child.stdout: print(line,end='',flush=True)
        status=child.wait()
        if status: raise RuntimeError(f'Lệnh thất bại, exit code {status}')
    except BaseException:
        child.terminate()
        raise
if not REPO.exists(): run_visible(['git','clone','--depth','1','https://github.com/hongphuc-git/wavelab-rl-scm.git',str(REPO)])
else: run_visible(['git','-C',str(REPO),'pull','--ff-only'])
print('Hoàn tất bước tải code.', flush=True)

In [ ]:
print('Kiểm tra GPU và PyTorch có sẵn...',flush=True)
import torch,torchvision
print('Torch:',torch.__version__,'Torchvision:',torchvision.__version__,flush=True)
assert torch.cuda.is_available(), 'Chọn Runtime → Change runtime type → GPU rồi kết nối lại'
print('GPU:',torch.cuda.get_device_name(0),flush=True)
print('Cài thư viện RL, log sẽ hiển thị bên dưới...',flush=True)
run_visible([sys.executable,'-m','pip','install','--disable-pip-version-check','--timeout','60','--retries','2','-r',str(REPO/'requirements-colab.txt')])
print('Cài đặt xong. Có thể chạy cell Cấu hình.',flush=True)

## Cấu hình
Epoch50 mặc định. Chạy từng EXPERIMENT rồi đánh giá trên cùng 32 mẫu; không đổi cấu hình khi train đang chạy.


In [ ]:
if 'process' in globals() and process.poll() is None: raise RuntimeError('Dừng train an toàn trước khi đổi nhánh')
if 'backup_thread' in globals() and backup_thread.is_alive(): raise RuntimeError('Chờ sao lưu hoàn tất trước khi đổi nhánh')
EPOCH=50
EXPERIMENT="physics_blur"  # physics_nearest, physics_le2024, parameter_progress
PRESETS={
 'physics_blur': ('absolute','blur',0.0),
 'physics_nearest': ('absolute','nearest',0.0),
 'physics_le2024': ('absolute','le2024',0.0),
 'parameter_progress': ('parameter_progress','blur',2.0),
}
REWARD_MODE,MISFIT_MODE,PARAM_WEIGHT=PRESETS[EXPERIMENT]
TOTAL_STEPS=4000
DRIVE=Path('/content/drive/MyDrive')
SOURCE=DRIVE/'conv/Bone_Phantom_Training_Data_b8.h5'
WEIGHTS=DRIVE/f'conv/18sep/pipe_efficientnet_v2_l/epoch_{EPOCH}_checkpoint/pytorch_model.bin'
DRIVE_RUN=DRIVE/f'WaveLab_RL/efficientnet_v2_l_epoch{EPOCH}/physics_compare_v1_256_{EXPERIMENT}_seed2026'
DRIVE_RUN.mkdir(parents=True,exist_ok=True)
RUN=Path(f'/content/rl_runs/epoch{EPOCH}_physics_compare_v1_256_{EXPERIMENT}')
RUN.mkdir(parents=True,exist_ok=True)
assert SOURCE.exists() and WEIGHTS.exists(), 'Kiểm tra đường dẫn Drive'
LOCAL_WEIGHTS=Path(f'/content/epoch{EPOCH}.bin')
if not LOCAL_WEIGHTS.exists(): shutil.copyfile(WEIGHTS,LOCAL_WEIGHTS)
os.environ['WAVEDL_ENCODER_CHECKPOINT']=str(LOCAL_WEIGHTS)
os.environ['RLSCM_BACKEND']='python'
for name in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'): os.environ[name]='1'
STOP=RUN/'STOP'
os.environ['RLSCM_STOP_FILE']=str(STOP)
# Restore only on a new local runtime; preserve matching replay metadata.
if not any(RUN.glob('checkpoint_*_steps.zip')):
    candidates=sorted(DRIVE_RUN.glob('checkpoint_*_steps.zip'),key=lambda p:int(p.stem.split('_')[1]))
    if candidates:
        newest=candidates[-1]
        assert (DRIVE_RUN/'run_config.json').exists(), 'Thiếu config để resume an toàn'
        shutil.copyfile(DRIVE_RUN/'run_config.json',RUN/'run_config.json')
        shutil.copyfile(newest,RUN/newest.name)
        meta=DRIVE_RUN/'replay_buffer.json'
        if meta.exists() and json.loads(meta.read_text()).get('step')==int(newest.stem.split('_')[1]):
            for name in ('replay_buffer.pkl','replay_buffer.json'):shutil.copyfile(DRIVE_RUN/name,RUN/name)
print('Train trên ổ cục bộ:',RUN,'; sao lưu:',DRIVE_RUN)

In [ ]:
import numpy as np,h5py
DATA=RUN/'train256.h5'
VALID=RUN/'validation32.h5'
if not DATA.exists():
    with h5py.File(SOURCE,'r') as f:
        counts=np.asarray(f['nnz_per_matrix']).ravel()
        ids=np.random.default_rng(20261113).choice(np.flatnonzero(counts>0),288,replace=False)
        labels=np.asarray(f['samples'])[:,ids].T
        H=int(np.asarray(f.attrs['grid_H']).item()); W=int(np.asarray(f.attrs['grid_W']).item())
        images=np.zeros((288,256,256),np.uint8)
        for j,i in enumerate(ids):
            n=int(counts[i]);r=f['row_indices'][i,:n].astype(np.int64);c=f['col_indices'][i,:n].astype(np.int64)
            images[j,np.minimum(r*256//H,255),np.minimum(c*256//W,255)]=1
        with h5py.File(DATA,'w') as g:
            g.attrs['split']='train';g.create_dataset('images',data=images[:256]);g.create_dataset('parameters',data=labels[:256]);g.create_dataset('complete',data=np.ones(256,bool));g.create_dataset('source_indices',data=ids[:256])
        with h5py.File(VALID,'w') as g:
            g.attrs['split']='validation';g.create_dataset('images',data=images[256:]);g.create_dataset('parameters',data=labels[256:]);g.create_dataset('complete',data=np.ones(32,bool));g.create_dataset('source_indices',data=ids[256:])
    manifest=dict(source=str(SOURCE),source_indices=ids.tolist(),geometry='free pipe b=8 mm',preprocessing='Original WaveDL floor mapping; binary 0/1 256x256',checkpoint=str(WEIGHTS),checkpoint_sha256=hashlib.sha256(LOCAL_WEIGHTS.read_bytes()).hexdigest(),diagnostic='256 train / 32 disjoint validation; one seed pilot; no independent test claim')
    (RUN/'manifest.json').write_text(json.dumps(manifest,indent=2))
print('Dataset:',DATA)

## Bắt đầu hoặc resume
Train chạy nền. Không chạy cell hai lần khi tiến trình còn sống. Mỗi 1k bước lưu model và replay vào Drive. Colab có thể mất kết nối; dùng cell này để resume.

In [ ]:
if 'process' in globals() and process.poll() is None: raise RuntimeError('Đang train')
assert SOURCE.exists() and WEIGHTS.exists(), 'Drive không truy cập được; kết nối lại Drive trước'
RUN.mkdir(parents=True,exist_ok=True)
probe=RUN/'write_probe.tmp'
probe.write_text('ok');probe.unlink()
print('Drive ghi được, bắt đầu train...',flush=True)
checkpoints=sorted(RUN.glob('checkpoint_*_steps.zip'),key=lambda p:int(p.stem.split('_')[1]))
latest=checkpoints[-1] if checkpoints else None
done=int(latest.stem.split('_')[1]) if latest else 0
assert done<TOTAL_STEPS, 'Đã đạt ngân sách'
STOP.unlink(missing_ok=True)
initial_lr=1e-4;final_lr=9.76e-5
resume_lr=initial_lr+(final_lr-initial_lr)*done/TOTAL_STEPS
cmd=[sys.executable,'-u',str(REPO/'scripts/train_bone_sac_parallel.py'),'--dataset',str(DATA),'--steps',str(TOTAL_STEPS-done),'--n-envs','4','--encoder',f'wavedl_epoch{EPOCH}','--share-encoder','--param-weight',str(PARAM_WEIGHT),'--misfit-mode',MISFIT_MODE,'--reward-mode',REWARD_MODE,'--horizon','8','--action-scale','0.15','--buffer-size','2000','--batch-size','8','--learning-starts','128','--learning-rate',str(resume_lr),'--lr-final',str(final_lr),'--save-every','1000','--save-replay','--seed','2026','--device','cuda','--torch-threads','2','--out',str(RUN)]
if latest:
    saved=json.loads((RUN/'run_config.json').read_text())
    sys.path.insert(0,str(REPO))
    from rlscm.reward_protocol import validate_resume
    validate_resume(saved,dict(reward_mode=REWARD_MODE,param_weight=PARAM_WEIGHT,misfit_mode=MISFIT_MODE,score_sigmas=[4.,12.]))
    cmd+=['--resume',str(latest)]
else: cmd+=['--entropy-coef','0.01','--save-initial']
session_log_offset=(RUN/'training.log').stat().st_size if (RUN/'training.log').exists() else 0
log=(RUN/'training.log').open('a')
process=subprocess.Popen(cmd,cwd=REPO,stdout=log,stderr=subprocess.STDOUT,env=os.environ.copy())
print('PID',process.pid,'resume từ',done,'mục tiêu',TOTAL_STEPS)
# Sao lưu ở thread riêng: chỉ công bố checkpoint sau khi copy hoàn tất.
import threading
backup_stop=threading.Event()
def copy_atomic(src,dst):
    temporary=dst.with_name(dst.name+'.partial')
    shutil.copyfile(src,temporary)
    temporary.replace(dst)
def backup_loop():
    while True:
        try:
            for cp in RUN.glob('checkpoint_*_steps.zip'):
                dest=DRIVE_RUN/cp.name
                # Checkpoint is written before replay; wait for completed save.
                if time.time()-cp.stat().st_mtime<30:continue
                import zipfile
                if not dest.exists() and zipfile.is_zipfile(cp):copy_atomic(cp,dest)
            meta=RUN/'replay_buffer.json'
            if meta.exists():
                before=meta.read_bytes()
                step=json.loads(before)['step']
                checkpoint=DRIVE_RUN/f'checkpoint_{step}_steps.zip'
                remote=DRIVE_RUN/'replay_buffer.json'
                old=json.loads(remote.read_text()).get('step') if remote.exists() else -1
                if checkpoint.exists() and old!=step:
                    copy_atomic(RUN/'replay_buffer.pkl',DRIVE_RUN/'replay_buffer.pkl')
                    if meta.read_bytes()==before:copy_atomic(meta,remote)
                    else:remote.unlink(missing_ok=True)
            for name in ('manifest.json','run_config.json','progress.json'):
                src=RUN/name
                if src.exists():copy_atomic(src,DRIVE_RUN/name)
            if process.poll() is not None:
                final=RUN/'model.zip'
                if final.exists():copy_atomic(final,DRIVE_RUN/'model.zip')
                if not list(RUN.glob('checkpoint_*_steps.zip')) or all((DRIVE_RUN/p.name).exists() for p in RUN.glob('checkpoint_*_steps.zip')):break
        except Exception as error:
            (RUN/'backup_error.txt').write_text(str(error))
        time.sleep(15)
backup_thread=threading.Thread(target=backup_loop,daemon=True)
backup_thread.start()
print('Sao lưu Drive chạy riêng; trước khi tắt runtime kiểm tra checkpoint và replay trên Drive.')


In [ ]:
# Bảng cố định: cập nhật tại chỗ, không clear_output toàn cell.
import time,json,html
from IPython.display import display,HTML
panel=display(HTML('<p>Đang đọc tiến độ...</p>'),display_id=True)
last_view=None
monitor_log=RUN/'training.log'
# Nếu cell launch cũ không ghi offset, chỉ đọc phiên OUTPUT gần nhất.
if 'session_log_offset' not in globals():
    tail=monitor_log.read_bytes() if monitor_log.exists() else b''
    marker=tail.rfind(b'OUTPUT:')
    session_log_offset=marker if marker>=0 else max(0,len(tail)-4000)
try:
    while True:
        progress_path=RUN/'progress.json'
        try: info=json.loads(progress_path.read_text()) if progress_path.exists() else {}
        except (OSError,ValueError): info={}
        status=process.poll() if 'process' in globals() else 'unknown'
        text=''
        if monitor_log.exists():
            with monitor_log.open('rb') as stream:
                stream.seek(min(session_log_offset,monitor_log.stat().st_size))
                text=stream.read().decode('utf-8',errors='replace')[-5000:]
        step=int(info.get('step',0));pct=min(100,100*step/TOTAL_STEPS)
        def fmt(value):return f'{value:.4f}' if isinstance(value,(int,float)) else '—'
        state='Đang chạy' if status is None else 'Không có tiến trình trong phiên notebook này' if status=='unknown' else 'Hoàn tất' if status==0 else 'Đã dừng an toàn' if status==3 else f'Đã dừng do lỗi · mã {status}'
        if status not in (None,0,3,'unknown') and 'KeyboardInterrupt' in text:
            state='Đã bị ngắt (KeyboardInterrupt)'
        J=info.get('mean_J');error=info.get('mean_param_error')
        reward=info.get('mean_reward')
        view=(step,info.get('critic_loss'),status)
        if view!=last_view:
            panel.update(HTML(f"""<div style="border:1px solid #438b91;border-radius:12px;padding:16px">
<h3>EfficientNet V2-L epoch {EPOCH} + SAC · {EXPERIMENT}</h3>
<progress value="{step}" max="{TOTAL_STEPS}" style="width:100%;height:24px"></progress>
<p><b>{step:,} / {TOTAL_STEPS:,} bước · {pct:.1f}% · {html.escape(state)}</b></p>
<p>J: {fmt(J)} · Reward: {fmt(reward)} · Sai số chuẩn hóa trong train: {fmt(error)}</p>
<p>Actor loss: {fmt(info.get('actor_loss'))} · Critic loss: {fmt(info.get('critic_loss'))}</p>
<p>Tốc độ: {fmt(info.get('env_steps_per_second'))} bước/giây · Còn khoảng {fmt(info.get('eta_hours')) if status is None else '—'} giờ</p>
<p>Log kỹ thuật được lưu trong training.log; không hiển thị trên bảng.</p>
</div>"""))
            last_view=view
        if status is not None:break
        time.sleep(3)
except KeyboardInterrupt:
    print('Dừng theo dõi; train vẫn chạy nền. Dùng STOP để dừng train an toàn.')


## Đánh giá và scatter trên 32 mẫu validation
Nhãn không truyền vào môi trường đánh giá. Lưu nghiệm cuối, best-J riêng của nhánh và nghiệm chọn theo blur-NCC chung.
Dùng cùng seed khởi tạo; scatter chính dùng tiêu chí chọn blur chung để so sánh công bằng. Sao chép cả CSV, JSON và PNG vào Drive.


In [ ]:
assert 'process' not in globals() or process.poll() is not None, 'Chờ train kết thúc'
run_visible([sys.executable,'-u',str(REPO/'scripts/evaluate_reward_physics.py'),
 '--run',str(RUN),'--dataset',str(VALID),'--device','cuda'])
shutil.copytree(RUN/'evaluation',DRIVE_RUN/'evaluation',dirs_exist_ok=True)
from IPython.display import Image,display
for image in sorted((RUN/'evaluation').glob('scatter_*.png')): display(Image(filename=str(image)))
print('Kết quả đã sao lưu:',DRIVE_RUN/'evaluation')


## Dừng an toàn
Chỉ chạy cell dưới khi muốn dừng. Chờ log STOPPED và tiến trình thoát trước khi tắt runtime.

In [ ]:
STOP.touch()
print('Đã yêu cầu lưu checkpoint + replay và dừng; kiểm tra log trước khi tắt runtime.')